# Week 1 · 三层框架与跨数据集可比性
本 Notebook 生成可复用的业务/ML 对照，不把未测量的变量当作没有作用，也不提前输出共通驱动因素。

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'config/study_design.json').exists())
sys.path.insert(0, str(ROOT/'src'))
from audit_utils import *
TABLES=ROOT/'outputs/tables'; FIGS=ROOT/'outputs/figures'
TABLES.mkdir(parents=True,exist_ok=True); FIGS.mkdir(parents=True,exist_ok=True)
pd.set_option('display.max_columns',20)
pd.set_option('display.width',160)
print('Project:', ROOT.name)


Project: Retention_Weeks1-4


In [2]:
mapping=pd.DataFrame([
 ['relationship_duration','tenure','Subscription  Length','days since first observed purchase (future feature)','Observed retail history is left-truncated; not true lifetime tenure'],
 ['engagement','service subscriptions only; usage intensity absent','Seconds of Use; Frequency of use; Frequency of SMS','recency_days; frequency_180d','Service enrollment is not behavioral frequency'],
 ['price_or_spending','MonthlyCharges; TotalCharges','Charge  Amount (ordinal)','gross_spend_180d','Spend is not price sensitivity; no common monetary scale'],
 ['service_friction','TechSupport subscription; not complaint events','Call  Failure; Complains','not directly measured','Support subscription is not dissatisfaction; returns need separate engineering'],
 ['contract_commitment','Contract','Tariff Plan','not applicable','Retail non-contractual purchases cannot be mapped to cancellation'],
 ['customer_value','historical TotalCharges proxy','Customer Value excluded until formula audit','gross historical spend proxy','None is validated future CLV or incremental treatment benefit'],
 ['retention_target','source Churn: Yes/No','source Churn: 1/0, nine-to-twelve-month design','future repeat purchase in 90 days','Different labels/horizons; compare process, not raw rates']
],columns=['concept','ibm','iranian','retail','comparability_limit'])
mapping.to_csv(TABLES/'feature_concept_map.csv',index=False)
print(mapping.to_string(index=False))


              concept                                                ibm                                            iranian                                              retail                                                            comparability_limit
relationship_duration                                             tenure                               Subscription  Length days since first observed purchase (future feature)            Observed retail history is left-truncated; not true lifetime tenure
           engagement service subscriptions only; usage intensity absent Seconds of Use; Frequency of use; Frequency of SMS                        recency_days; frequency_180d                                 Service enrollment is not behavioral frequency
    price_or_spending                       MonthlyCharges; TotalCharges                           Charge  Amount (ordinal)                                    gross_spend_180d                       Spend is not price sensitivity; no co

In [3]:
framework=pd.DataFrame([
 ['1 Business definition','eligible cohort, outcome, horizon, action objective','Source labels vs retail purchase windows; business assumptions remain explicit','target_definition.md + study_design.json'],
 ['2 ML analysis and validation','sample/feature construction, leakage checks, baseline comparison, calibration, robustness','Separate models and suitable splits; frozen holdouts; no model fitted Week 1','notebooks 01/02 + split/window files'],
 ['3 Business application and feedback','cohort retention, risk/value prioritization, experiments, monitoring','10% outreach is a later scenario; no observed treatment effects or financial savings','decision_log.md + Week 2 handoff']
],columns=['layer','reusable_logic','adaptation_or_boundary','week1_output'])
framework.to_csv(TABLES/'three_layer_framework.csv',index=False)
print(framework.to_string(index=False))


                              layer                                                                            reusable_logic                                                               adaptation_or_boundary                             week1_output
              1 Business definition                                       eligible cohort, outcome, horizon, action objective       Source labels vs retail purchase windows; business assumptions remain explicit target_definition.md + study_design.json
       2 ML analysis and validation sample/feature construction, leakage checks, baseline comparison, calibration, robustness         Separate models and suitable splits; frozen holdouts; no model fitted Week 1     notebooks 01/02 + split/window files
3 Business application and feedback                      cohort retention, risk/value prioritization, experiments, monitoring 10% outreach is a later scenario; no observed treatment effects or financial savings         decision_log.md + Week 2 

## 同一个 ML 框架如何接受不同数据？
统一接口：`build_dataset(raw, design) -> X, y, customer_group, cutoff, split`。

- X 只使用 cutoff 之前可获得的字段；标签、ID、未来交易和 outcome-derived 字段不进入 X。
- 每个数据集分别拟合 preprocessing/model/calibration；retail 用时间验证，静态 telecom 仅验证同来源泛化。
- 后续统一报告 ROC-AUC、average precision、calibration/Brier、precision/recall/lift at capacity，以及 cohort retention error。
- 跨场景比较的是方法的增益、概念组稳定性和失败条件；未经归一的 SHAP 数值不跨模型硬比较。

In [4]:
inventory=pd.read_csv(TABLES/'dataset_inventory.csv')
quality=pd.read_csv(TABLES/'field_quality_profile.csv')
assert set(inventory.dataset)=={'ibm','iranian','retail'}
assert all((TABLES/n).exists() for n in ['retail_window_plan.csv','ibm_split_assignments.csv','iranian_split_assignments.csv'])
summary={'datasets_acquired':3,'raw_rows':inventory.set_index('dataset').rows.to_dict(),
 'field_profiles':len(quality),'mapping_concepts':len(mapping),'models_fitted':0,
 'test_performance_computed':False,'retail_test_labels_constructed':False,
 'common_drivers_established':False,'status':'Week 1 data/task feasibility completed; unresolved definitions documented'}
(ROOT/'outputs/week1_status.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


{
  "datasets_acquired": 3,
  "raw_rows": {
    "ibm": 7043,
    "iranian": 3150,
    "retail": 1067371
  },
  "field_profiles": 43,
  "mapping_concepts": 7,
  "models_fitted": 0,
  "test_performance_computed": false,
  "retail_test_labels_constructed": false,
  "common_drivers_established": false,
  "status": "Week 1 data/task feasibility completed; unresolved definitions documented"
}


## 下周优先事项
1. 确认 Iranian Status/客户 eligibility 与 Customer Value 公式，验证分组策略。
2. 确认零售调整单/商品行与退货口径，做开发期敏感性检查。
3. 在 development 内创建 fold-safe preprocessing 和 Logistic Regression 基线。
4. 记录目标、划分或字段变更；测试集继续保留，不据其表现修改方法。